# NSL-KDD Dataset Preprocessing

This notebook downloads and preprocesses the **NSL-KDD** dataset from Kaggle
(`hassan06/nslkdd`).

The dataset is an improved version of KDD Cup 1999 and contains network connection
records with **41 features**, one `attack_type` label, and a `difficulty` column.
Files are headerless `.txt` CSVs split into train and test sets.

**Attack categories**
- `normal`
- DoS: `back`, `land`, `neptune`, `pod`, `smurf`, `teardrop`, …
- Probe: `ipsweep`, `nmap`, `portsweep`, `satan`, …
- R2L: `ftp_write`, `guess_passwd`, `imap`, `multihop`, `phf`, …
- U2R: `buffer_overflow`, `loadmodule`, `perl`, `rootkit`, …

**Pipeline**
1. Download via `kagglehub`
2. Load with explicit column names (no header in source files)
3. Clean data (NaNs, duplicates, `difficulty` column drop)
4. Encode categorical features and attack label
5. Add a coarse 5-class `attack_category` column (normal / DoS / Probe / R2L / U2R)
6. Normalise numerical features
7. Save train and test outputs separately
8. Download outputs (Colab)


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('Libraries imported.')

## 1. Download the dataset

In [ ]:
print('Downloading dataset...')
data_path = kagglehub.dataset_download('hassan06/nslkdd')
print(f'Dataset ready at: {data_path}')

print('\nFiles in dataset:')
for f in sorted(os.listdir(data_path)):
    print(f'  {f}')

## 2. Configuration

In [ ]:
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# NSL-KDD has no header — column names must be supplied manually.
# 41 features + attack_type + difficulty (42nd and 43rd columns)
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes',
    'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack_type',   # fine-grained label  (e.g. 'neptune', 'normal')
    'difficulty'     # will be dropped
]

# The three categorical feature columns
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']

TARGET_COL    = 'attack_type'     # fine-grained (24+ classes)
CATEGORY_COL  = 'attack_category' # coarse 5-class column we will add

# Map every specific attack name → its broad category
ATTACK_CATEGORY_MAP = {
    'normal':            'normal',
    # DoS
    'back':              'dos', 'land':          'dos', 'neptune':   'dos',
    'pod':               'dos', 'smurf':         'dos', 'teardrop':  'dos',
    'apache2':           'dos', 'udpstorm':      'dos', 'processtable': 'dos',
    'worm':              'dos', 'mailbomb':      'dos',
    # Probe
    'ipsweep':           'probe', 'nmap':        'probe', 'portsweep': 'probe',
    'satan':             'probe', 'mscan':       'probe', 'saint':     'probe',
    # R2L
    'ftp_write':         'r2l', 'guess_passwd': 'r2l', 'imap':       'r2l',
    'multihop':          'r2l', 'phf':          'r2l', 'spy':        'r2l',
    'warezclient':       'r2l', 'warezmaster':  'r2l', 'sendmail':   'r2l',
    'named':             'r2l', 'snmpgetattack':'r2l', 'snmpguess':  'r2l',
    'xlock':             'r2l', 'xsnoop':       'r2l', 'httptunnel': 'r2l',
    # U2R
    'buffer_overflow':   'u2r', 'loadmodule':   'u2r', 'perl':       'u2r',
    'rootkit':           'u2r', 'ps':           'u2r', 'sqlattack':  'u2r',
    'xterm':             'u2r', 'snmpguess':    'u2r',
}

# File pairs: (input filename, output base name, split role)
FILE_PAIRS = [
    ('KDDTrain+.txt', 'nslkdd_train', 'train'),
    ('KDDTest+.txt',  'nslkdd_test',  'test'),
]

print('Config set.')

## 3. Preprocessing functions

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Drop difficulty column, fix NaNs, remove exact duplicates."""
    df = df.copy()

    # Drop the difficulty score — not a predictive feature
    if 'difficulty' in df.columns:
        df.drop(columns=['difficulty'], inplace=True)

    # Replace inf with NaN and fill numeric NaNs with column median
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    num_cols = df.select_dtypes(include='number').columns.tolist()
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # Normalise attack_type: lowercase + strip
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.lower()

    # Remove exact duplicate rows
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'  Duplicates removed: {before - len(df):,}')

    return df


def add_attack_category(df: pd.DataFrame) -> pd.DataFrame:
    """Add a coarse 5-class attack_category column."""
    df = df.copy()
    df[CATEGORY_COL] = df[TARGET_COL].map(ATTACK_CATEGORY_MAP).fillna('unknown')
    return df


def downcast_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce memory usage by downcasting numeric columns."""
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


print('Functions defined.')

## 4. Fit encoders and scaler on training data

In [ ]:
label_encoders: dict         = {}   # one LabelEncoder per categorical col
attack_encoder: LabelEncoder = LabelEncoder()
category_encoder: LabelEncoder = LabelEncoder()
scaler: StandardScaler       = StandardScaler()
numeric_cols: list           = []


def fit_transformers(df: pd.DataFrame) -> None:
    """Fit all encoders and the scaler on the training dataframe."""
    global numeric_cols

    # Categorical feature encoders
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        le.fit(df[col].astype(str))
        label_encoders[col] = le
        print(f'  [{col}] classes: {list(le.classes_)}')

    # Fine-grained attack label encoder
    attack_encoder.fit(df[TARGET_COL])
    print(f'  [attack_type] classes ({len(attack_encoder.classes_)}): {list(attack_encoder.classes_)}')

    # Coarse category encoder
    category_encoder.fit(df[CATEGORY_COL])
    print(f'  [attack_category] classes: {list(category_encoder.classes_)}')

    # Numeric columns for scaling (everything except label/category columns)
    exclude = CATEGORICAL_COLS + [TARGET_COL, CATEGORY_COL]
    numeric_cols = [c for c in df.columns if c not in exclude]

    scaler.fit(df[numeric_cols])
    print(f'  Scaler fitted on {len(numeric_cols)} numeric columns.')


def encode_and_scale(df: pd.DataFrame) -> pd.DataFrame:
    """Apply fitted transformers to a dataframe (train or test)."""
    df = df.copy()

    # Encode categorical features — unseen values fall back to first class
    for col in CATEGORICAL_COLS:
        le = label_encoders[col]
        df[col] = df[col].astype(str).apply(
            lambda x: x if x in le.classes_ else le.classes_[0]
        )
        df[col] = le.transform(df[col])

    # Encode fine-grained attack label
    df[TARGET_COL] = df[TARGET_COL].apply(
        lambda x: x if x in attack_encoder.classes_ else 'normal'
    )
    df[TARGET_COL] = attack_encoder.transform(df[TARGET_COL])

    # Encode coarse category
    df[CATEGORY_COL] = df[CATEGORY_COL].apply(
        lambda x: x if x in category_encoder.classes_ else 'unknown'
    )
    df[CATEGORY_COL] = category_encoder.transform(df[CATEGORY_COL])

    # Scale numeric features
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    return df


print('Transformer helpers defined.')

## 5. Process train and test files

In [ ]:
fitted = False

for filename, out_base, role in FILE_PAIRS:
    path = os.path.join(data_path, filename)
    print(f'\nProcessing {role} file: {filename}')

    # ── Load (no header in source files) ────────────────────────────────────
    df = pd.read_csv(path, header=None, names=COL_NAMES)
    print(f'  Loaded: {len(df):,} rows, {df.shape[1]} columns')

    # ── Clean ───────────────────────────────────────────────────────────────
    df = clean_data(df)
    df = add_attack_category(df)
    df = downcast_dtypes(df)
    print(f'  After cleaning: {len(df):,} rows')

    # ── Fit only on training data ────────────────────────────────────────────
    if not fitted:
        print('  Fitting encoders and scaler on training data...')
        fit_transformers(df)
        fitted = True

    df = encode_and_scale(df)

    # ── Save ─────────────────────────────────────────────────────────────────
    csv_out  = os.path.join(OUTPUT_DIR, f'{out_base}_processed.csv')
    json_out = os.path.join(OUTPUT_DIR, f'{out_base}_processed.jsonl')

    df.to_csv(csv_out, index=False)
    df.to_json(json_out, orient='records', lines=True)

    print(f'  ✅  Saved {len(df):,} rows → {csv_out}')
    del df

print(f'\n{"="*55}')
print('✅  Processing complete!')

## 6. Quick sanity check

In [ ]:
train_sample = pd.read_csv(
    os.path.join(OUTPUT_DIR, 'nslkdd_train_processed.csv'), nrows=5
)
print('Train — first 5 rows:')
print(train_sample.head())
print()
print('Dtypes:')
print(train_sample.dtypes)

print('\nAttack-type classes (fine-grained):')
for i, cls in enumerate(attack_encoder.classes_):
    print(f'  {i:2d} → {cls}')

print('\nAttack-category classes (coarse):')
for i, cls in enumerate(category_encoder.classes_):
    print(f'  {i} → {cls}')

## 7. Download outputs (Colab only)

In [ ]:
output_files = [
    os.path.join(OUTPUT_DIR, 'nslkdd_train_processed.csv'),
    os.path.join(OUTPUT_DIR, 'nslkdd_train_processed.jsonl'),
    os.path.join(OUTPUT_DIR, 'nslkdd_test_processed.csv'),
    os.path.join(OUTPUT_DIR, 'nslkdd_test_processed.jsonl'),
]

try:
    from google.colab import files
    for f in output_files:
        print(f'Downloading {os.path.basename(f)}...')
        files.download(f)
except ImportError:
    print('Not running in Colab — files saved locally:')
    for f in output_files:
        print(f'  {f}')